#### RAG with PRO Techniques

- RAG with Advanced Techinques

    1. No LangChain! Just native for maximum flexibility

    2. Use LLM to divide the chunks in a sensible way

    3. Use LLM to re-write chunks in a way that's most useful ("document pre-processing)



In [ ]:
# imports

import os
from dotenv import load_dotenv
from pathlib import Path
from litellm import completion
from pydantic import BaseModel, Field                       # pydantic way of structured outputs
from chromadb import PersistentClient                       # ChromaDB client library
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE                           # projecting chunks to 2D or 3D
import plotly.graph_objects as go

In [2]:
load_dotenv(override=True)

ollama_base_url = 'http://localhost:11434/'
ollama_api_key = os.getenv('OLLAMA_API_KEY')

MODEL = 'ollama/llama3.2'
DB_NAME = 'preprocessed_db'
collection_name = 'docs'
embedding_model = 'all-MiniLM-L6-v2'
KNOWLEDGE_BASE_PATH = Path('../knowledge-base')
AVERAGE_CHUNK_SIZE = 500


In [3]:
# For RAG friendly result just like LangChain

class Result(BaseModel):
    page_content : str
    metadata : dict

In [4]:
# class to perfectly represent a chunk

class Chunk(BaseModel):
    headline : str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary : str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    orginal_text : str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {'source': document['source'], 'type':document['type']}
        return Result(
            page_content=self.headline + '\n\n' + self.summary + '\n\n' + self.orginal_text, metadata=metadata
        )

In [5]:
# class to represent the entire data as a list of Chunk

class Chunks(BaseModel):
    chunks : list[Chunk]

#### Three Steps:

1. Fetch documents from the knowledge base, like LangChain did

2. Call an LLM to turn documents into Chunks

3. Store the Chunks in Chroma

In [6]:
# Step 1
# to fetch the md files from knowledge base into list of dictionaries

def fetch_documents():
    """A homemade version of the LangChain Directory loader"""
    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob('*.md'):
            with open(file, 'r', encoding='utf-8') as f:
                documents.append({'type':doc_type, 'source':file.as_posix(), 'text':f.read()})

    print(f'Loaded {len(documents)} documents')
    return documents

''' 
`file.as_posix()`   - returns individual file path using /

'''

' \n`file.as_posix()`   - returns individual file path using /\n\n'

In [7]:
documents = fetch_documents()
documents[0]

Loaded 76 documents


{'type': 'products',
 'source': '../knowledge-base/products/Rellm.md',
 'text': "# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seamless In

In [8]:
# Step 2 - use llm to convert the data into efficient chunks for retrieval
# to make a prompt

def make_prompt(document):
    ''' Creates a prompt to ask the llm to convert the data into efficient chunks for retrieval '''
    how_many = (len(document['text']) // AVERAGE_CHUNK_SIZE) + 1
    return f""" 
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [9]:
print(make_prompt(documents[0]))

 
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: products
The document has been retrieved from: ../knowledge-base/products/Rellm.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 8 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Product Summary

# Rellm: AI-Powered Enterpr

In [10]:
# to convert the prompt into user_prompt

def make_message(document):
    return [
        {'role':'user', 'content':make_prompt(document)}
    ]

In [11]:
make_message(documents[0])

[{'role': 'user',
  'content': " \nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: ../knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 8 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\

In [12]:
# using llm to process the document into chunks

def process_documents(document):
    messages = make_message(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks, base_url=ollama_base_url, api_key=ollama_api_key)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

''' 
The LLM generates the response in JSON format.

Then, the JSON output is validated against the `Chunks` Pydantic model and converted into a `Chunks` object. From that object, only the list of `Chunk` objects is extracted.

Then, each `Chunk` in that list is converted into a `Result` object using `chunk.as_result(document)`.

Finally, the entire list of `Result` objects is returned.

'''

' \nThe LLM generates the response in JSON format.\n\nThen, the JSON output is validated against the `Chunks` Pydantic model and converted into a `Chunks` object. From that object, only the list of `Chunk` objects is extracted.\n\nThen, each `Chunk` in that list is converted into a `Result` object using `chunk.as_result(document)`.\n\nFinally, the entire list of `Result` objects is returned.\n\n'

- `response_format=Chunks`            - returns the response as Chunks object in JSON format


      {
        "chunks": [
          {
            "headline": "Visa Requirements",
            "summary": "This section explains the required documents.",
            "original_text": "Applicants must submit a valid passport."
          },
          {
            "headline": "Application Process",
            "summary": "This section explains how to apply.",
            "original_text": "Applications can be submitted online."
          }
        ]
      }



- `Chunks.model_validate_json(reply)`   - validates the json format and converts it to Chunks object i.e., list[Chunk]

      Chunks(
          chunks=[
              Chunk(
                  headline="Visa Requirements",
                  summary="This section explains the required documents.",
                  original_text="Applicants must submit a valid passport."
              ),
              Chunk(
                  headline="Application Process",
                  summary="This section explains how to apply.",
                  original_text="Applications can be submitted online."
              )
          ]
      )


- `Chunks.model_validate_json(reply).chunks`  - extracts only the list[Chunk] from the previous output

      [
          Chunk(
              headline="Visa Requirements",
              summary="This section explains the required documents.",
              original_text="Applicants must submit a valid passport."
          ),
          Chunk(
              headline="Application Process",
              summary="This section explains how to apply.",
              original_text="Applications can be submitted online."
          )
      ]


- `[chunk.as_result(document) for chunk in doc_as_chunks]`    - converts each Chunk object into Result object and returns the output as list of Result objects

      [
          Result(
              page_content="Headline + Summary + Original Text",
              metadata={"source": "...", "type": "..."}
          ),
          Result(
              page_content="Headline + Summary + Original Text",
              metadata={"source": "...", "type": "..."}
          )
      ]

In [13]:
process_documents(documents[0])

[Result(page_content='Rellm: AI-Powered Enterprise Reinsurance Solution\n\nInnovative enterprise reinsurance product developed by Insurellm, harnessing artificial intelligence to redefine risk management and optimize operational efficiencies.\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry.', metadata={'source': '../knowledge-base/products/Rellm.md', 'type': 'products'}),
 Result(page_content='Key Features\n\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions.\n\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes

In [14]:
# to create the chunks for entire documents data

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm


def create_chunks(documents, max_workers=5):

    chunks = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:       # multi threading

        results = executor.map(process_documents, documents)

        for result in tqdm(results, total=len(documents)):              # tqdm gives the progress bar showing how far the loop is progressed
            chunks.extend(result)

    return chunks

In [ ]:
chunks = create_chunks(documents, max_workers=5)

 46%|████▌     | 35/76 [53:28<1:02:38, 91.66s/it] 



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



In [ ]:
# Step 3 - Creating Vector DB

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    embedding_model = SentenceTransformer("sentence_transformers/all-MiniLM-L6-v2")
    vectors = embedding_model.encode(texts).tolist()

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f'VectorStore created with {collection.count()} documents')
    